In [ ]:
import pandas as pd
from meta_project.data.data_loader import DataLoader
data_loader = DataLoader()

df = data_loader.load_and_merge_data()
df.head()

In [ ]:
import torch
from torch.utils import data
from PIL import Image
import numpy as np

class ImageDataset(data.Dataset):

    def __init__(self, imgs, targets, img_transform=None):
        super().__init__()
        self.img_transform = img_transform
        self.imgs = imgs
        self.targets = targets

    def __getitem__(self, idx):
        img, target = self.imgs[idx], self.targets[idx]
        img = Image.fromarray(img)

        if self.img_transform is not None:
            img = self.img_transform(img)

        return img, target

    def __len__(self):
        return self.imgs.shape[0]

In [ ]:
import torch

torch.manual_seed(0)

unique_classes = df['class_id'].unique()
randomized_classes = torch.randperm(len(unique_classes))

train_classes = unique_classes[randomized_classes[:160]]
val_classes = unique_classes[randomized_classes[160:180]]
test_classes = unique_classes[randomized_classes[180:]]

train_df = df[df['class_id'].isin(train_classes)]
val_df = df[df['class_id'].isin(val_classes)]
test_df = df[df['class_id'].isin(test_classes)]

print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")


In [ ]:
import os

IMAGE_FOLDER = os.getcwd() + "/data/raw/CUB_200_2011/images/"

def dataset_from_labels(imgs, targets, class_set, **kwargs):
    class_mask = (targets[:,None] == class_set[None,:]).any(dim=-1)
    return ImageDataset(imgs=imgs[class_mask],
                        targets=targets[class_mask],
                        **kwargs)
def load_data_from_df(dataframe, image_size=(224, 224)):
    imgs = []
    targets = []
    
    for idx, row in dataframe.iterrows():
        img_path = os.path.join(IMAGE_FOLDER, row['image_name'])
        img = Image.open(img_path)

        img = img.convert('RGB')
        img = img.resize(image_size)

        imgs.append(np.array(img))
        targets.append(row['class_id'])
    
    return np.array(imgs), torch.tensor(targets)

train_imgs, train_targets = load_data_from_df(train_df)
val_imgs, val_targets = load_data_from_df(val_df)
test_imgs, test_targets = load_data_from_df(test_df)

In [ ]:
import torch
from torchvision import transforms

# Pre-computed statistics (use your own statistics here)
DATA_MEANS = torch.Tensor([0.5183975, 0.49192241, 0.44651328])
DATA_STD = torch.Tensor([0.26770132, 0.25828985, 0.27961241])

# Transformations for testing and training
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Adjust to the desired image size
    transforms.ToTensor(),
    transforms.Normalize(DATA_MEANS, DATA_STD)
])

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(DATA_MEANS, DATA_STD)
])

# Create the datasets from the CUB-200-2011 dataset
train_set = dataset_from_labels(train_imgs, train_targets, train_classes, img_transform=train_transform)
val_set = dataset_from_labels(val_imgs, val_targets, val_classes, img_transform=test_transform)
test_set = dataset_from_labels(test_imgs, test_targets, test_classes, img_transform=test_transform)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir models/MAML/

C:\Users\Michal\Desktop\MAML\metaLearningCUB-200-2011
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6007 (pid 14656), started 0:00:02 ago. (Use '!kill 14656' to kill it.)